In [1]:
!pip install -q -U ultralytics roboflow

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("sohamblulabel")  # re-add this secret if it's a fresh notebook

import torch
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow

ultralytics.checks()

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Notebook Settings → Accelerator → GPU T4 x2, then restart."
    )
print(f"GPU available: {torch.cuda.get_device_name(0)}")

rf = Roboflow(api_key="vnmgAvGE6feVLByShlGT")
project = rf.workspace("soham-bhattacharya-mwcvr").project("button-mic")
version = project.version(1)
dataset = version.download("yolov11")
                
print(f"\nDataset downloaded to: {dataset.location}")

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)
GPU available: Tesla T4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Button-Mic-1 in yolov11:: 100%|██████████| 407/407 [00:00<00:00, 2862.74it/s]



Dataset downloaded to: /kaggle/working/Button-Mic-1


In [5]:
import os
import yaml

DATASET_PATH = dataset.location
data_yaml_path = os.path.join(DATASET_PATH, "data.yaml")

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

fixed_config = dict(data_config)
for split_key, folder_name in [("train", "train"), ("val", "valid"), ("test", "test")]:
    img_dir = os.path.join(DATASET_PATH, folder_name, "images")
    if os.path.isdir(img_dir):
        fixed_config[split_key] = img_dir
fixed_config["path"] = DATASET_PATH

fixed_yaml_path = "/kaggle/working/data.yaml"
with open(fixed_yaml_path, "w") as f:
    yaml.dump(fixed_config, f, default_flow_style=False)

class_names = fixed_config.get("names", [])
print(f"Classes detected: {class_names}")

Classes detected: ['defective', 'passing']


In [8]:
fixed_config = dict(data_config)
for split_key, folder_name in [("train", "train"), ("val", "valid"), ("test", "test")]:
    img_dir = os.path.join(DATASET_PATH, folder_name, "images")
    if os.path.isdir(img_dir):
        fixed_config[split_key] = img_dir
    else:
        print(f"NOTE: {img_dir} not found — removing '{split_key}' from config")
        fixed_config.pop(split_key, None)
fixed_config["path"] = DATASET_PATH

NOTE: /kaggle/working/Button-Mic-1/valid/images not found — removing 'val' from config
NOTE: /kaggle/working/Button-Mic-1/test/images not found — removing 'test' from config


In [10]:
from collections import Counter

def gather_pairs(img_dir):
    label_dir = img_dir.replace("images", "labels")
    pairs = []
    for fname in sorted(os.listdir(img_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        img_path = os.path.join(img_dir, fname)
        lbl_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")
        pairs.append((img_path, lbl_path, fname))
    return pairs

all_pairs = []
for split_key in ["train", "val", "test"]:
    if split_key in fixed_config:
        all_pairs.extend(gather_pairs(fixed_config[split_key]))
print(f"Total images pooled: {len(all_pairs)}")

def get_class_for_image(lbl_path):
    """Assumes one connector class per image, per this project's labeling scheme."""
    if not os.path.exists(lbl_path) or os.path.getsize(lbl_path) == 0:
        return None
    with open(lbl_path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    if not lines:
        return None
    class_idx = int(lines[0].split()[0])
    return class_names[class_idx] if 0 <= class_idx < len(class_names) else None

def get_color_for_image(fname):
    """ADJUST THIS if your white images used a different naming convention —
    verify against the printed counts below before trusting it."""
    return "white" if "white" in fname.lower() else "black"

tagged_pairs = []
for img_path, lbl_path, fname in all_pairs:
    cls = get_class_for_image(lbl_path)
    if cls is None:
        continue
    tagged_pairs.append((img_path, lbl_path, cls, get_color_for_image(fname)))

group_counts = Counter((cls, color) for _, _, cls, color in tagged_pairs)
print("Group counts (class, color) — verify these match expectations:")
for group, count in sorted(group_counts.items()):
    print(f"  {group}: {count}")

Total images pooled: 202
Group counts (class, color) — verify these match expectations:
  ('defective', 'black'): 101
  ('passing', 'black'): 101


In [11]:
from sklearn.model_selection import train_test_split

group_key = [f"{cls}_{color}" for _, _, cls, color in tagged_pairs]

train_pairs, temp_pairs = train_test_split(
    tagged_pairs, test_size=0.30, stratify=group_key, random_state=42
)
temp_group_key = [f"{cls}_{color}" for _, _, cls, color in temp_pairs]
test_pairs, val_pairs = train_test_split(
    temp_pairs, test_size=(10/30), stratify=temp_group_key, random_state=42
)

print(f"Train: {len(train_pairs)} | Test (20%): {len(test_pairs)} | Val (10%): {len(val_pairs)}")
for name, pairs in [("train", train_pairs), ("test", test_pairs), ("val", val_pairs)]:
    counts = Counter((c, col) for _, _, c, col in pairs)
    print(f"  {name}: {dict(counts)}")

Train: 141 | Test (20%): 40 | Val (10%): 21
  train: {('defective', 'black'): 70, ('passing', 'black'): 71}
  test: {('passing', 'black'): 20, ('defective', 'black'): 20}
  val: {('passing', 'black'): 10, ('defective', 'black'): 11}


In [12]:
import cv2
import shutil

ZOOMED_ROOT = "/kaggle/working/zoomed_dataset"
ZOOM_FACTOR = 2.0
MAX_DIM = 1280
MIN_BOX_FRACTION = 0.15

if os.path.exists(ZOOMED_ROOT):
    shutil.rmtree(ZOOMED_ROOT)  # clear stale output — this is what fixed the v3 leakage incident

def adjust_and_write_labels(lbl_path, out_lbl_path, orig_w, orig_h, crop_w, crop_h, x0, y0, min_frac):
    if not os.path.exists(lbl_path):
        open(out_lbl_path, "w").close()
        return
    with open(lbl_path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    out_lines = []
    for line in lines:
        parts = line.split()
        cls_id = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])

        abs_xc, abs_yc = xc * orig_w, yc * orig_h
        abs_bw, abs_bh = bw * orig_w, bh * orig_h
        x1, y1 = abs_xc - abs_bw / 2, abs_yc - abs_bh / 2
        x2, y2 = abs_xc + abs_bw / 2, abs_yc + abs_bh / 2

        orig_area = (x2 - x1) * (y2 - y1)
        x1c, y1c = max(x1 - x0, 0), max(y1 - y0, 0)
        x2c, y2c = min(x2 - x0, crop_w), min(y2 - y0, crop_h)

        if x2c <= x1c or y2c <= y1c:
            continue
        if (x2c - x1c) * (y2c - y1c) / orig_area < min_frac:
            continue

        new_bw, new_bh = (x2c - x1c) / crop_w, (y2c - y1c) / crop_h
        new_xc, new_yc = (x1c + x2c) / 2 / crop_w, (y1c + y2c) / 2 / crop_h
        out_lines.append(f"{cls_id} {new_xc:.6f} {new_yc:.6f} {new_bw:.6f} {new_bh:.6f}")

    with open(out_lbl_path, "w") as f:
        f.write("\n".join(out_lines))

for split_name, pairs in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
    img_out_dir = os.path.join(ZOOMED_ROOT, split_name, "images")
    lbl_out_dir = os.path.join(ZOOMED_ROOT, split_name, "labels")
    os.makedirs(img_out_dir, exist_ok=True)
    os.makedirs(lbl_out_dir, exist_ok=True)

    for img_path, lbl_path, cls, color in sorted(pairs, key=lambda p: p[0]):
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        crop_w, crop_h = w / ZOOM_FACTOR, h / ZOOM_FACTOR
        x0, y0 = (w - crop_w) / 2, (h - crop_h) / 2
        cropped = img[int(y0):int(y0 + crop_h), int(x0):int(x0 + crop_w)]

        scale = min(1.0, MAX_DIM / max(cropped.shape[:2]))
        if scale < 1.0:
            cropped = cv2.resize(cropped, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

        base_name = os.path.splitext(os.path.basename(img_path))[0]
        out_name = f"{color}__{base_name}"
        cv2.imwrite(os.path.join(img_out_dir, out_name + ".jpg"), cropped)
        adjust_and_write_labels(
            lbl_path, os.path.join(lbl_out_dir, out_name + ".txt"),
            w, h, crop_w, crop_h, x0, y0, MIN_BOX_FRACTION
        )

zoomed_yaml_path = "/kaggle/working/zoomed_data.yaml"
zoomed_config = {
    "path": ZOOMED_ROOT,
    "train": os.path.join(ZOOMED_ROOT, "train", "images"),
    "val": os.path.join(ZOOMED_ROOT, "val", "images"),
    "test": os.path.join(ZOOMED_ROOT, "test", "images"),
    "nc": len(class_names),
    "names": class_names,
}
with open(zoomed_yaml_path, "w") as f:
    yaml.dump(zoomed_config, f, default_flow_style=False)
print(f"Zoomed dataset written to: {zoomed_yaml_path}")

Zoomed dataset written to: /kaggle/working/zoomed_data.yaml


In [13]:
import albumentations as A

augment_pipeline = A.Compose([
    A.Rotate(limit=8, border_mode=cv2.BORDER_REPLICATE, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.7),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.5),
    A.GaussNoise(std_range=(0.01, 0.03), p=0.3),
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.4))

train_img_dir = os.path.join(ZOOMED_ROOT, "train", "images")
train_lbl_dir = os.path.join(ZOOMED_ROOT, "train", "labels")

for fname in list(os.listdir(train_img_dir)):
    if "_aug" in fname:
        os.remove(os.path.join(train_img_dir, fname))
for fname in list(os.listdir(train_lbl_dir)):
    if "_aug" in fname:
        os.remove(os.path.join(train_lbl_dir, fname))

base_images = sorted(f for f in os.listdir(train_img_dir) if "_aug" not in f)

for fname in base_images:
    base_name = os.path.splitext(fname)[0]
    img = cv2.imread(os.path.join(train_img_dir, fname))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    lbl_path = os.path.join(train_lbl_dir, base_name + ".txt")
    bboxes, class_labels = [], []
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            for line in f:
                if line.strip():
                    parts = line.split()
                    class_labels.append(int(float(parts[0])))
                    bboxes.append([float(x) for x in parts[1:5]])

    for aug_idx in range(2):
        augmented = augment_pipeline(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
        aug_img = cv2.cvtColor(augmented["image"], cv2.COLOR_RGB2BGR)
        out_name = f"{base_name}_aug{aug_idx}"
        cv2.imwrite(os.path.join(train_img_dir, out_name + ".jpg"), aug_img)
        with open(os.path.join(train_lbl_dir, out_name + ".txt"), "w") as f:
            for cls, box in zip(augmented["class_labels"], augmented["bboxes"]):
                f.write(f"{int(cls)} {' '.join(f'{v:.6f}' for v in box)}\n")

print(f"Train images after 3x augmentation: {len(os.listdir(train_img_dir))}")

Train images after 3x augmentation: 423


In [14]:
import re

def get_source_id(filename):
    name = os.path.splitext(filename)[0]
    return re.sub(r"_aug\d+$", "", name)

split_ids = {
    split: set(get_source_id(f) for f in os.listdir(os.path.join(ZOOMED_ROOT, split, "images")))
    for split in ["train", "val", "test"]
}
overlaps = {f"{a}/{b}": split_ids[a] & split_ids[b] for a, b in [("train","val"),("train","test"),("val","test")]}

if any(overlaps.values()):
    for pair, ov in overlaps.items():
        if ov:
            print(f"LEAKAGE in {pair}: {len(ov)} overlapping images")
    raise RuntimeError("DATA LEAKAGE DETECTED — refusing to proceed to training.")
print("No leakage detected — safe to proceed to training.")

No leakage detected — safe to proceed to training.


In [15]:
from ultralytics import YOLO

# Train from COCO-pretrained weights — this is a different object than the
# connector project, so there's no existing checkpoint worth fine-tuning from.
model = YOLO("yolo11n.pt")

results = model.train(
    data=zoomed_yaml_path,
    imgsz=640,
    epochs=100,
    batch=16,
    patience=20,
    device=0,
    project="/kaggle/working/runs",
    name="aoi_mic_wiring_v1",
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    seed=42,
    fliplr=0.0,   # still required — same reasoning as the connector: a flip
                  # would show red on the physically-impossible side
)
print("Best weights: /kaggle/working/runs/aoi_mic_wiring_v1/weights/best.pt")

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/zoomed_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=aoi_mic_wiring_v1, nbs=64, nm

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      2.39G      1.198      2.086      1.298         15        640: 100% ━━━━━━━━━━━━ 27/27 7.4it/s 3.6s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.2it/s 0.3s
                   all         21         21      0.405          1      0.865      0.542

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.39G      1.059      1.681      1.248         14        640: 100% ━━━━━━━━━━━━ 27/27 7.0it/s 3.9s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.4it/s 0.3s
                   all         21         21      0.916      0.828      0.918       0.57

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      2.39G      1.055      1.467      1.248         11        640: 100% ━━━━━━━━━━━━ 27/27 7.0it/s 3.9s0.1s
                 Class     Images  Instances      Box(

In [19]:
# Load the best checkpoint explicitly to confirm it saved correctly
best_model = YOLO("/kaggle/working/runs/aoi_mic_wiring_v1/weights/best.pt")

# --- Validation set metrics (same split used during training) ---
val_metrics = best_model.val(data=zoomed_yaml_path, imgsz=640, device=0, split="val")

print("\n=== Validation Metrics (seen during training loop) ===")
print(f"mAP50:    {val_metrics.box.map50:.4f}")
print(f"mAP50-95: {val_metrics.box.map:.4f}")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")

# --- Test set metrics (the model NEVER saw this during training or early stopping) ---
test_metrics = best_model.val(data=zoomed_yaml_path, imgsz=640, device=0, split="test")

print("\n=== Test Metrics (true unbiased check) ===")
print(f"mAP50:    {test_metrics.box.map50:.4f}")
print(f"mAP50-95: {test_metrics.box.map:.4f}")
print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall:    {test_metrics.box.mr:.4f}")

# Per-class breakdown on test set — this is the number that matters most for QC
print("\n=== Per-Class AP50 (Test Set) ===")
for i, class_name in enumerate(class_names):
    if i < len(test_metrics.box.ap50):
        print(f"  Class '{class_name}': AP50 = {test_metrics.box.ap50[i]:.4f}")

Ultralytics 8.4.138 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2748.7±597.4 MB/s, size: 117.6 KB)
val: Scanning /kaggle/working/zoomed_dataset/val/labels.cache... 21 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21/21 8.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.6it/s 0.6s1.2s
                   all         21         21      0.963      0.955      0.988      0.928
             defective         11         11      0.955      0.909      0.981      0.903
               passing         10         10       0.97          1      0.995      0.953
Speed: 4.6ms preprocess, 4.2ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-22

=== Validation Metrics (seen during training loop) ===
mAP50:    0.9881
mAP50

In [20]:
import shutil

# Bundle everything you need off Kaggle into one zip: weights + training plots + confusion matrix
output_dir = "/kaggle/working/runs/aoi_mic_wiring_v1"
shutil.make_archive("/kaggle/working/aoi_mic_wiring_results", "zip", output_dir)

print("Download this file from the Kaggle 'Output' tab:")
print("  aoi_mic_wiring_results.zip")
print("\nInside you'll find:")
print("  weights/best.pt      <- this is what goes on the Raspberry Pi")
print("  weights/last.pt      <- final epoch checkpoint (backup)")
print("  confusion_matrix.png <- check this before trusting mAP")
print("  results.png          <- loss/mAP curves over training")
print("  val_batch*.jpg       <- sample predictions on validation images")

Download this file from the Kaggle 'Output' tab:
  aoi_mic_wiring_results.zip

Inside you'll find:
  weights/best.pt      <- this is what goes on the Raspberry Pi
  weights/last.pt      <- final epoch checkpoint (backup)
  confusion_matrix.png <- check this before trusting mAP
  results.png          <- loss/mAP curves over training
  val_batch*.jpg       <- sample predictions on validation images
